<a href="https://colab.research.google.com/github/FengruiJing/TrafficVideoVisualEnvironment/blob/main/TrafficVideoVisualEnv1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install torch torchvision matplotlib numpy opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 53.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [2]:
#mount google drive
from google.colab import drive
drive.mount('/content/gdrive')
import os
os.chdir("/content/gdrive/My Drive/Colab Notebooks")
#list the files
!ls

Mounted at /content/gdrive
 5DeepSouth_HIV_Merged.cpg
 5DeepSouth_HIV_Merged.dbf
 5DeepSouth_HIV_Merged.prj
 5DeepSouth_HIV_Merged.shp
 5DeepSouth_HIV_Merged.shx
 A0927.gpx
 activity_area_results20241231.csv
 activity_area_results.csv
 activity_radius_results.csv
 activity_radius_results_in_meters.csv
 ALL_VARIAIBLES2-1-coord1.xlsx
 ALL_VARIAIBLES2-1-coord.xlsx
 ALL_VARIAIBLES2-1.csv
 ALL_VARIAIBLES2-1.xlsx
 awesome-semantic-segmentation-pytorch
 B0928.gpx
 C0928.gpx
 CA_HIV_Merged.cpg
 CA_HIV_Merged.dbf
 CA_HIV_Merged.prj
 CA_HIV_Merged.shp
 CA_HIV_Merged.shx
 Categorical_Variable_Summary_gender.xlsx
 Categorical_Variable_Summary_Park.xlsx
 CBSA_HIV_Merged.cpg
 CBSA_HIV_Merged.dbf
 CBSA_HIV_Merged.prj
 CBSA_HIV_Merged.shp
 CBSA_HIV_Merged.shx
 Community_variables_finnal_test_tree.xlsx
 Community_variables_finnal_test.xlsx
 Community_variables_with_water_combinations_MultplBy.xlsx
 Community_variables_with_water_combinations.xlsx
'Copy of SpatialRegressionV2.ipynb'
'Copy of SpatialRegr

In [ ]:
#Video
import torch
from torchvision import models, transforms
import cv2
import numpy as np
import pandas as pd

#######################
# 1. 加载预训练的 Mask R-CNN 模型
#######################
model = models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")
model.eval()

#######################
# 2. 预处理
#######################
def preprocess_image(image):
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    transform_fn = transforms.Compose([transforms.ToTensor()])
    image_tensor = transform_fn(image_rgb).unsqueeze(0)  # (1, C, H, W)
    return image_tensor

#######################
# 3. 目标检测
#######################
@torch.no_grad()
def detect_objects(image):
    image_tensor = preprocess_image(image)
    prediction = model(image_tensor)
    return prediction

#######################
# 4. 空间拥堵度计算 (SCi)
#######################
def calculate_space_congestion(image, boxes, grid_size=(50, 50)):
    """
    计算空间拥堵度 (SCi)。
    """
    height, width, _ = image.shape
    num_grids_x = width // grid_size[0]
    num_grids_y = height // grid_size[1]
    if num_grids_x == 0 or num_grids_y == 0:
        return 0

    grid_counts = np.zeros((num_grids_y, num_grids_x))
    for box in boxes:
        x1, y1, x2, y2 = box
        gx1, gy1 = int(x1 // grid_size[0]), int(y1 // grid_size[1])
        gx2, gy2 = int(x2 // grid_size[0]), int(y2 // grid_size[1])

        gx1 = max(0, gx1)
        gy1 = max(0, gy1)
        gx2 = min(num_grids_x - 1, gx2)
        gy2 = min(num_grids_y - 1, gy2)

        grid_counts[gy1:gy2 + 1, gx1:gx2 + 1] += 1

    total_density = grid_counts.sum() / (num_grids_x * num_grids_y)
    max_count = np.max(grid_counts)
    if max_count == 0:
        return 0

    congestion_score = total_density / max_count
    return congestion_score

#######################
# 5. 视觉复杂度计算 (VisualComplex)
#######################
def calculate_visual_complexity(frame):
    """
    使用灰度图熵(Entropy)衡量视觉复杂度。
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
    hist_norm = hist.ravel() / hist.sum()
    epsilon = 1e-7
    entropy = -np.sum(hist_norm * np.log2(hist_norm + epsilon))
    return entropy

#######################
# 6. 最小距离计算 (MinDist)
#######################
def calculate_min_distance(boxes):
    """
    计算所有目标之间的最小距离。
    """
    if len(boxes) < 2:
        return float('inf')

    centers = [((b[0] + b[2]) / 2, (b[1] + b[3]) / 2) for b in boxes]
    min_dist = float('inf'
                    )
    for i in range(len(centers)):
        for j in range(i + 1, len(centers)):
            dist = np.linalg.norm(np.array(centers[i]) - np.array(centers[j]))
            min_dist = min(min_dist, dist)
    return min_dist

#######################
# 7. 新混行严重度计算 (New Mxi)
#######################
def calculate_new_mxi(image, prediction):
    """
    计算基于不同交通主体类型的新的混行严重度。
    """
    boxes = prediction[0]['boxes'].cpu().numpy()
    labels = prediction[0]['labels'].cpu().numpy()
    scores = prediction[0]['scores'].cpu().numpy()

    threshold = 0.5
    valid_indices = scores >= threshold
    boxes = boxes[valid_indices]
    labels = labels[valid_indices]

    type_weights = {
        (1, 2): 2.0, (1, 3): 1.5, (1, 4): 2.5,
        (2, 3): 1.5, (2, 4): 2.0, (3, 4): 1.8,
        (1, 1): 1.0, (2, 2): 1.0, (3, 3): 1.0,
        (4, 4): 1.2
    }

    interaction_sum = 0
    total_pairs = 0
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            type_i = labels[i]
            type_j = labels[j]
            dist = np.linalg.norm(np.array([(boxes[i][0] + boxes[i][2]) / 2, (boxes[i][1] + boxes[i][3]) / 2]) -
                                  np.array([(boxes[j][0] + boxes[j][2]) / 2, (boxes[j][1] + boxes[j][3]) / 2]))
            weight = type_weights.get((min(type_i, type_j), max(type_i, type_j)), 1.0)
            interaction_sum += weight / (dist + 1e-5)
            total_pairs += 1

    mxi_new = interaction_sum / total_pairs if total_pairs > 0 else 0
    return mxi_new

#######################
# 8. 综合近距冲突频次计算
#######################
def compute_time_conflict_frequency(mxi_values, min_distances, mxi_threshold, dist_threshold, time_interval=1.0):
    """
    综合基于 New Mxi 和 MinDist 计算近距冲突频次。
    """
    conflict_count = sum(
        1 for mxi, dist in zip(mxi_values, min_distances)
        if mxi > mxi_threshold or dist < dist_threshold
    )
    total_seconds = len(mxi_values) * time_interval
    conflicts_per_second = conflict_count / total_seconds if total_seconds > 0 else 0
    return conflicts_per_second

#######################
# 9. 视频分析
#######################
def analyze_video(video_path, output_path=None, mxi_threshold=0.005, dist_threshold=50, time_interval=1.0):
    cap = cv2.VideoCapture(video_path)
    results = []
    mxi_values = []
    min_distances = []

    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    else:
        out = None

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        prediction = detect_objects(frame)
        boxes = prediction[0]['boxes'].cpu().numpy() if len(prediction[0]['boxes']) else []
        sc = calculate_space_congestion(frame, boxes)
        mxi = calculate_new_mxi(frame, prediction)
        vc = calculate_visual_complexity(frame)
        min_dist = calculate_min_distance(boxes)

        mxi_values.append(mxi)
        min_distances.append(min_dist)

        results.append((frame_idx, sc, mxi, vc, min_dist))

        if out:
            for box in boxes:
                x1, y1, x2, y2 = map(int, box)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            out.write(frame)

        print(f"Frame {frame_idx}: SCi={sc:.3f}, New Mxi={mxi:.3f}, VisualComplex={vc:.3f}, MinDist={min_dist:.3f}")
        frame_idx += 1

    near_conflict_freq = compute_time_conflict_frequency(
        mxi_values, min_distances, mxi_threshold, dist_threshold, time_interval
    )
    print(f"\n=== 视频近距冲突频次: {near_conflict_freq:.3f} 次/秒 ===")

    df = pd.DataFrame(results, columns=['Frame', 'SCi', 'New Mxi', 'VisualComplex', 'MinDist'])
    df.to_csv('video_analysis_results.csv', index=False)
    print("结果已保存: video_analysis_results.csv")

    cap.release()
    if out:
        out.release()
    return df, near_conflict_freq

#######################
# 10. 主程序调用
#######################
if __name__ == "__main__":
    video_path = 'videoplayback2.mp4'
    output_path = 'output_video1.avi'

    df, conflict_per_second = analyze_video(video_path, output_path)

    print("\n分析结束.")
    print(df)
    print(f"近距冲突频次(次/秒): {conflict_per_second:.3f}")

In [4]:
#Pictures
import torch
from torchvision import models, transforms
import cv2
import numpy as np
import pandas as pd

#######################
# 1. 加载预训练的 Mask R-CNN 模型
#######################
model = models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")
model.eval()

#######################
# 2. 预处理
#######################
def preprocess_image(image):
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    transform_fn = transforms.Compose([transforms.ToTensor()])
    image_tensor = transform_fn(image_rgb).unsqueeze(0)  # (1, C, H, W)
    return image_tensor

#######################
# 3. 目标检测
#######################
@torch.no_grad()
def detect_objects(image):
    image_tensor = preprocess_image(image)
    prediction = model(image_tensor)
    return prediction

#######################
# 4. 新混行严重度计算
#######################
def calculate_new_mxi(image, prediction):
    """
    计算基于不同交通主体类型的新的混行严重度。
    """
    # 提取目标框、类型和置信度
    boxes = prediction[0]['boxes'].cpu().numpy()
    labels = prediction[0]['labels'].cpu().numpy()  # 类别标签
    scores = prediction[0]['scores'].cpu().numpy()

    # 筛选高置信度目标
    threshold = 0.5
    valid_indices = scores >= threshold
    boxes = boxes[valid_indices]
    labels = labels[valid_indices]

    # 类型间权重矩阵（可以根据实际情况调整）
    type_weights = {
        (1, 2): 2.0,  # 行人与汽车
        (1, 3): 1.5,  # 行人与自行车
        (1, 4): 2.5,  # 行人与电动自行车
        (2, 3): 1.5,  # 汽车与自行车
        (2, 4): 2.0,  # 汽车与电动自行车
        (3, 4): 1.8,  # 自行车与电动自行车
        (1, 1): 1.0,  # 行人之间
        (2, 2): 1.0,  # 汽车之间
        (3, 3): 1.0,  # 自行车之间
        (4, 4): 1.2   # 电动自行车之间
    }

    # 计算类型间的交互影响
    interaction_sum = 0
    total_pairs = 0

    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            type_i = labels[i]
            type_j = labels[j]
            dist = np.linalg.norm(np.array([(boxes[i][0] + boxes[i][2]) / 2, (boxes[i][1] + boxes[i][3]) / 2]) -
                                  np.array([(boxes[j][0] + boxes[j][2]) / 2, (boxes[j][1] + boxes[j][3]) / 2]))
            weight = type_weights.get((min(type_i, type_j), max(type_i, type_j)), 1.0)
            interaction_sum += weight / (dist + 1e-5)
            total_pairs += 1

    # 计算新的混行程度
    mxi_new = interaction_sum / total_pairs if total_pairs > 0 else 0
    return mxi_new

#######################
# 5. 最小距离计算
#######################
def calculate_min_distance(boxes):
    """
    计算所有目标之间的最小距离。
    """
    if len(boxes) < 2:
        return float('inf')

    centers = [((b[0] + b[2]) / 2, (b[1] + b[3]) / 2) for b in boxes]
    min_dist = float('inf')
    for i in range(len(centers)):
        for j in range(i + 1, len(centers)):
            dist = np.linalg.norm(np.array(centers[i]) - np.array(centers[j]))
            min_dist = min(min_dist, dist)
    return min_dist

#######################
# 6. 综合近距冲突频次计算
#######################
def compute_time_conflict_frequency(mxi_values, min_distances, mxi_threshold, dist_threshold, time_interval=2.0):
    """
    综合基于 Mxi 和最小距离计算近距冲突频次。
    """
    conflict_count = sum(
        1 for mxi, dist in zip(mxi_values, min_distances)
        if mxi > mxi_threshold or dist < dist_threshold
    )

    n_frames = len(mxi_values)
    if n_frames > 1:
        total_seconds = (n_frames - 1) * time_interval
    else:
        total_seconds = time_interval

    conflicts_per_second = conflict_count / total_seconds if total_seconds > 0 else 0
    return conflicts_per_second

#######################
# 7. 主函数: 多张图片模拟时间序列
#######################
def analyze_images_as_time_sequence(image_paths, mxi_threshold=0.005, dist_threshold=50, time_interval=2.0):
    """
    分析多张图片的时间序列，计算 SCi、New Mxi、VisualComplex 和 Near-Conflict Frequency。
    """
    results = []
    mxi_values = []
    min_distances = []

    for idx, img_path in enumerate(image_paths):
        frame = cv2.imread(img_path)
        if frame is None:
            print(f"无法读取图像 {img_path}")
            continue

        # 目标检测
        prediction = detect_objects(frame)

        # 计算指标
        boxes = prediction[0]['boxes'].cpu().numpy() if len(prediction[0]['boxes']) else []
        sc = calculate_space_congestion(frame, boxes)
        mxi = calculate_new_mxi(frame, prediction)
        vc = calculate_visual_complexity(frame)
        min_dist = calculate_min_distance(boxes)

        # 保存 Mxi 和最小距离用于时间序列分析
        mxi_values.append(mxi)
        min_distances.append(min_dist)

        # 输出当前帧结果
        print(f"Frame {idx} ({img_path}):")
        print(f"  SCi={sc:.3f}, New Mxi={mxi:.3f}, VisualComplex={vc:.3f}, MinDist={min_dist:.3f}")

        results.append((idx, img_path, sc, mxi, vc, min_dist))

    # 计算综合近距冲突频次
    near_conflict_freq = compute_time_conflict_frequency(
        mxi_values, min_distances, mxi_threshold, dist_threshold, time_interval
    )
    print(f"\n=== 多张图像近距冲突频次: {near_conflict_freq:.3f} 次/秒 ===")

    # 保存到CSV
    df = pd.DataFrame(results, columns=['Frame', 'ImagePath', 'SCi', 'New Mxi', 'VisualComplex', 'MinDist'])
    df.to_csv('multiple_images_results.csv', index=False)
    print("结果已保存: multiple_images_results.csv")

    return df, near_conflict_freq

#######################
# 8. 使用示例
#######################
if __name__ == "__main__":
    # 假设有三张图片, 间隔2秒
    image_list = [
        "Road2.png",
        "Road3.png",
        "Road4.png"
    ]

    # 运行分析
    df, conflict_per_second = analyze_images_as_time_sequence(
        image_paths=image_list,
        mxi_threshold=0.005,       # 冲突事件的 Mxi 阈值
        dist_threshold=50,        # 冲突事件的最小距离阈值
        time_interval=2.0         # 2 秒间隔
    )

    print("\n分析结束.")
    print(df)
    print(f"近距冲突频次(次/秒): {conflict_per_second:.3f}")

Frame 0 (Road2.png):
  SCi=0.069, New Mxi=0.005, VisualComplex=7.817, MinDist=0.077
Frame 1 (Road3.png):
  SCi=0.071, New Mxi=0.006, VisualComplex=7.787, MinDist=0.371
Frame 2 (Road4.png):
  SCi=0.096, New Mxi=0.005, VisualComplex=7.842, MinDist=0.234

=== 多张图像近距冲突频次: 0.750 次/秒 ===
结果已保存: multiple_images_results.csv

分析结束.
   Frame  ImagePath       SCi   New Mxi  VisualComplex   MinDist
0      0  Road2.png  0.068982  0.005500       7.816743  0.077468
1      1  Road3.png  0.070530  0.005539       7.787105  0.370516
2      2  Road4.png  0.095553  0.004840       7.842158  0.234353
近距冲突频次(次/秒): 0.750
